# Task 4: Time-Slice Permutation Feature Importance (PFI) & Sanity Check
## Machine Learning & AI Case Study: Arrhythmia Detection & Classification

**Task Ownership: Group Member 4**

### Objectives:
1. Implement the **Time-Slice Permutation Feature Importance (PFI)** algorithm:
   - $K=5$ stratified cross-validation folds.
   - 11 non-overlapping time slices (25 samples per slice, ~69.4 ms each).
   - Within-slice random permutation (`axis=1`) independently per beat.
   - Compute 0-1 misclassification error delta: $\Delta = \text{Err}_{\text{perm}} - \text{Err}_{\text{base}}$.
2. Execute the **Planted Signal Sanity Check** on synthetic data with an isolated QRS impulse in Slice 6 to verify algorithm correctness.
3. Compute empirical PFI vectors for both models across both splits:
   - `SVM_beat_holdout`, `SVM_patient_holdout`, `RF_beat_holdout`, `RF_patient_holdout`.
4. Export all four importance score vectors to `.npy` files for report generation and cross-team use.


In [1]:
# Step 1: Library Imports
import os
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from scipy.stats import spearmanr

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 150

print("Libraries imported successfully.")


Libraries imported successfully.


### Step 2: Data Loading & Separation of Signals, Labels, and Patient IDs

Each ECG heartbeat sample has **277 columns**:
- Columns `0` to `274`: **275 ECG signal amplitude points** (centered at R-peak ~ index 137).
- Column `275`: **Class Label** (numeric 1.0 to 8.0).
- Column `276`: **Patient Number** (e.g., 100 to 234).

We load the beat-holdout and patient-holdout CSVs and safely separate features from metadata.


In [2]:
# Mapping of numeric identifiers to cardiology beat classes
CLASS_MAPPING = {
    1: 'Normal (N)',
    2: 'LBBB (L)',
    3: 'RBBB (R)',
    4: 'PVC (V)',
    5: 'APB (A)',
    6: 'Fusion Normal/Ventricular (F)',
    7: 'Fusion Paced/Normal (f)',
    8: 'Paced Beat (/)'
}

CLASS_NAMES = ['N', 'L', 'R', 'V', 'A', 'F', 'f', '/']

def load_data(train_path, test_path, n_time=275):
    """
    Safely load ECG CSV data, separating 275 signal features from labels and patient IDs.
    """
    print(f"Loading {os.path.basename(train_path)} and {os.path.basename(test_path)}...")
    tr = pd.read_csv(train_path, header=None).values
    te = pd.read_csv(test_path, header=None).values
    
    X_train = tr[:, :n_time].astype(np.float32)
    y_train = tr[:, n_time].astype(int)
    
    X_test  = te[:, :n_time].astype(np.float32)
    y_test  = te[:, n_time].astype(int)
    
    pat_train = tr[:, n_time + 1].astype(int) if tr.shape[1] > n_time + 1 else None
    pat_test  = te[:, n_time + 1].astype(int) if te.shape[1] > n_time + 1 else None
    
    print(f"  Train: X={X_train.shape}, y={y_train.shape}")
    print(f"  Test:  X={X_test.shape}, y={y_test.shape}")
    return X_train, y_train, X_test, y_test, pat_train, pat_test

# Load both split datasets
X_tr_b, y_tr_b, X_te_b, y_te_b, pat_tr_b, pat_te_b = load_data('train_beats.csv', 'test_beats.csv')
X_tr_p, y_tr_p, X_te_p, y_te_p, pat_tr_p, pat_te_p = load_data('train_patients.csv', 'test_patients.csv')


Loading train_beats.csv and test_beats.csv...
  Train: X=(30992, 275), y=(30992,)
  Test:  X=(37442, 275), y=(37442,)
Loading train_patients.csv and test_patients.csv...
  Train: X=(200352, 275), y=(200352,)
  Test:  X=(14482, 275), y=(14482,)


### Step 3: Class Distribution Inspection
Let's inspect the class balances in the training sets (balanced through resampling) and the test sets.


In [3]:
# Inspect class distribution
def print_class_distribution(y, title):
    classes, counts = np.unique(y, return_counts=True)
    print(f"--- {title} ---")
    for c, count in zip(classes, counts):
        print(f"  Class {c} ({CLASS_MAPPING.get(c, 'Unknown')}): {count} samples ({count/len(y)*100:.2f}%)")
    print(f"  Total: {len(y)} samples\n")

print_class_distribution(y_tr_b, "Beat-Holdout Train Set (Resampled)")
print_class_distribution(y_te_b, "Beat-Holdout Test Set (Natural Distribution)")
print_class_distribution(y_tr_p, "Patient-Holdout Train Set (Resampled)")
print_class_distribution(y_te_p, "Patient-Holdout Test Set (Unseen Patients: 104, 208, 113, 210, 119)")


--- Beat-Holdout Train Set (Resampled) ---
  Class 1 (Normal (N)): 3874 samples (12.50%)
  Class 2 (LBBB (L)): 3874 samples (12.50%)
  Class 3 (RBBB (R)): 3874 samples (12.50%)
  Class 4 (PVC (V)): 3874 samples (12.50%)
  Class 5 (APB (A)): 3874 samples (12.50%)
  Class 6 (Fusion Normal/Ventricular (F)): 3874 samples (12.50%)
  Class 7 (Fusion Paced/Normal (f)): 3874 samples (12.50%)
  Class 8 (Paced Beat (/)): 3874 samples (12.50%)
  Total: 30992 samples

--- Beat-Holdout Test Set (Natural Distribution) ---
  Class 1 (Normal (N)): 28474 samples (76.05%)
  Class 2 (LBBB (L)): 1636 samples (4.37%)
  Class 3 (RBBB (R)): 2581 samples (6.89%)
  Class 4 (PVC (V)): 1307 samples (3.49%)
  Class 5 (APB (A)): 783 samples (2.09%)
  Class 6 (Fusion Normal/Ventricular (F)): 191 samples (0.51%)
  Class 7 (Fusion Paced/Normal (f)): 416 samples (1.11%)
  Class 8 (Paced Beat (/)): 2054 samples (5.49%)
  Total: 37442 samples

--- Patient-Holdout Train Set (Resampled) ---
  Class 1 (Normal (N)): 25044 s

### Step 4: Stratified Subsampling Function
Because SVM has quadratic/cubic complexity with sample size ($O(n^2)$ to $O(n^3)$), training on the full 100k+ beats is computationally prohibitive.
As instructed in the case study guide, we use **stratified subsampling** to preserve exact class ratios while speeding up model fitting and permutation evaluation.


In [4]:
def stratified_subsample(X, y, n=5000, seed=42):
    """
    Draw a stratified subsample preserving exact class proportions.
    """
    if n >= len(X):
        return X, y
    X_sub, _, y_sub, _ = train_test_split(
        X, y,
        train_size=n,
        stratify=y,
        random_state=seed
    )
    return X_sub, y_sub

# Prepare subsamples for training (N=8000 for training, N=3000 for 5-fold PFI)
X_train_b_sub, y_train_b_sub = stratified_subsample(X_tr_b, y_tr_b, n=8000, seed=42)
X_train_p_sub, y_train_p_sub = stratified_subsample(X_tr_p, y_tr_p, n=8000, seed=42)

# Smaller subsamples for Permutation Feature Importance (5 folds * 11 slices = 55 predictions)
X_pfi_b, y_pfi_b = stratified_subsample(X_tr_b, y_tr_b, n=3000, seed=42)
X_pfi_p, y_pfi_p = stratified_subsample(X_tr_p, y_tr_p, n=3000, seed=42)

print(f"Subsampled Beat-holdout train for modeling: {X_train_b_sub.shape}")
print(f"Subsampled Patient-holdout train for modeling: {X_train_p_sub.shape}")


Subsampled Beat-holdout train for modeling: (8000, 275)
Subsampled Patient-holdout train for modeling: (8000, 275)


## Step 7: Permutation Feature Importance (PFI) Implementation

### Algorithm & Specification (from Assignment Task Guide):
1. Split data into **$K = 5$ stratified folds**.
2. For each fold $k$:
   - Train a fresh clone of the classifier on the other 4 folds.
   - Compute baseline 0-1 misclassification error on fold $k$: $\text{Err}_{\text{base}} = \frac{1}{N_k} \sum \mathbb{I}(\hat{y} \neq y)$.
   - For each of the **11 slices** (25 time points each, totaling 275 samples):
     - Perturb the slice: **randomly shuffle the time points within the slice for each beat independently** (`rng.permuted(X_te[:, sl], axis=1)`).
     - Compute the permuted misclassification error $\text{Err}_{\text{perm}}$.
     - Difference $\Delta = \text{Err}_{\text{perm}} - \text{Err}_{\text{base}}$ measures importance of that slice.
3. Average $\Delta$ over all folds to yield the final importance vector $\mathbf{I} \in \mathbb{R}^{11}$.


In [8]:
def perm_importance(model, X, y, n_splits=5, slice_size=25, n_repeats=1, seed=42):
    """
    Permutation feature importance by time slice with stratified K-fold cross-validation.
    
    Parameters:
    -----------
    model      : Scikit-learn classifier (unfitted; cloned in each fold)
    X          : (n_samples, 275) array of ECG signals
    y          : (n_samples,) array of integer class labels (1 to 8)
    n_splits   : Number of stratified folds (default 5)
    slice_size : Number of time points per slice (default 25; 11 slices total)
    n_repeats  : Number of permutation repeats to average per fold
    seed       : Random state for reproducibility
    
    Returns:
    --------
    mean_imp   : (11,) array of mean importance scores per slice
    I_matrix   : (n_samples, 11) full sample-level importance difference matrix
    """
    rng = np.random.default_rng(seed)
    n_slices = X.shape[1] // slice_size  # 275 // 25 = 11 slices
    I_matrix = np.zeros((len(X), n_slices))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    print(f"Running PFI ({n_splits}-fold CV, {n_slices} slices of {slice_size} pts, N={len(X)})...")
    t0 = time.time()
    
    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
        # Fit fresh model on 4 folds
        m = clone(model).fit(X[tr_idx], y[tr_idx])
        X_te, y_te = X[te_idx], y[te_idx]
        
        # Baseline 0-1 misclassification error
        base_err = (m.predict(X_te) != y_te).astype(float)
        
        for j in range(n_slices):
            sl = slice(j * slice_size, (j + 1) * slice_size)
            diffs = np.zeros(len(te_idx))
            
            for _ in range(n_repeats):
                X_perm = X_te.copy()
                # Random shuffle time points within slice for each beat independently
                X_perm[:, sl] = rng.permuted(X_te[:, sl], axis=1)
                perm_err = (m.predict(X_perm) != y_te).astype(float)
                diffs += (perm_err - base_err)
                
            I_matrix[te_idx, j] = diffs / n_repeats
            
        print(f"  Fold {fold}/{n_splits} complete.")
        
    mean_imp = I_matrix.mean(axis=0)
    print(f"PFI finished in {time.time() - t0:.1f}s.")
    return mean_imp, I_matrix


### Step 8: Planted Signal Sanity Check
Before running on real ECG data, we run a controlled synthetic check where class 1 differs with a distinct wave peak **exclusively in Slice 6** (samples 125 to 150).
The function should find that **Slice 6 has by far the highest importance**, while all other slices remain near zero.


In [9]:
# Controlled sanity check
rng = np.random.default_rng(1)
n_synth = 800
y_fake = rng.integers(0, 2, n_synth)
X_fake = rng.normal(0, 0.5, (n_synth, 275))

# Plant a distinctive peak only in Slice 6 (cols 125 to 150)
# Plant a distinctive sharp QRS-like waveform in Slice 6 (cols 125 to 150)
pattern = np.zeros(25)
pattern[10:15] = [2.0, 5.0, 8.0, 5.0, 2.0]
pattern[7:10] = [-1.0, -2.0, -1.0]
pattern[15:18] = [-1.0, -2.0, -1.0]
X_fake[y_fake == 1, 125:150] += pattern
# Plant a distinctive sharp QRS-like waveform in Slice 6 (cols 125 to 150)
pattern = np.zeros(25)
pattern[10:15] = [2.0, 5.0, 8.0, 5.0, 2.0]
pattern[7:10] = [-1.0, -2.0, -1.0]
pattern[15:18] = [-1.0, -2.0, -1.0]
X_fake[y_fake == 1, 125:150] += pattern

imp_test, _ = perm_importance(SVC(kernel='rbf', C=10.0), X_fake, y_fake, n_splits=3)

print("Sanity Check Slice Scores:")
for idx, s in enumerate(imp_test, start=1):
    print(f"  Slice {idx:2d}: {s:.4f}")

top_slice = np.argmax(imp_test) + 1
print(f"\nMost important slice: Slice {top_slice} (Expected: 6)")
assert top_slice == 6, "Sanity check failed: Slice 6 was not identified as top slice!"
print(">>> SANITY CHECK PASSED! Function correctly identifies feature signal location. <<<")


Running PFI (3-fold CV, 11 slices of 25 pts, N=800)...
  Fold 1/3 complete.
  Fold 2/3 complete.
  Fold 3/3 complete.
PFI finished in 0.2s.
Sanity Check Slice Scores:
  Slice  1: 0.0000
  Slice  2: 0.0000
  Slice  3: 0.0000
  Slice  4: 0.0000
  Slice  5: 0.0000
  Slice  6: 0.4700
  Slice  7: 0.0000
  Slice  8: 0.0000
  Slice  9: 0.0000
  Slice 10: 0.0000
  Slice 11: 0.0000

Most important slice: Slice 6 (Expected: 6)
>>> SANITY CHECK PASSED! Function correctly identifies feature signal location. <<<


## Step 9: Compute Permutation Feature Importance on MIT-BIH Arrhythmias
Now we compute the 11-slice importance scores for:
1. `SVM_beat_holdout`
2. `SVM_patient_holdout`
3. `RF_beat_holdout`
4. `RF_patient_holdout`

All importance vectors are saved to `.npy` files for report generation and cross-team sharing.


In [10]:
# Run PFI experiments
pfi_experiments = {
    'SVM_beat_holdout': (SVC(kernel='rbf', C=10.0, random_state=42), X_pfi_b, y_pfi_b),
    'SVM_patient_holdout': (SVC(kernel='rbf', C=10.0, random_state=42), X_pfi_p, y_pfi_p),
    'RF_beat_holdout': (RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1), X_pfi_b, y_pfi_b),
    'RF_patient_holdout': (RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1), X_pfi_p, y_pfi_p)
}

importance_results = {}

for name, (clf, X_data, y_data) in pfi_experiments.items():
    print(f"\n=================== Computing {name} ===================")
    imp_mean, I_full = perm_importance(clf, X_data, y_data, n_splits=5, slice_size=25, seed=42)
    importance_results[name] = imp_mean
    
    # Save .npy file
    np.save(f"imp_{name}.npy", imp_mean)
    print(f"Saved imp_{name}.npy with scores: {np.round(imp_mean, 4)}")



=================== Computing SVM_beat_holdout ===================
Running PFI (5-fold CV, 11 slices of 25 pts, N=3000)...
  Fold 1/5 complete.
  Fold 2/5 complete.
  Fold 3/5 complete.
  Fold 4/5 complete.
  Fold 5/5 complete.
PFI finished in 4.7s.
Saved imp_SVM_beat_holdout.npy with scores: [0.007  0.0007 0.0007 0.0007 0.013  0.5547 0.0993 0.0017 0.001  0.001
 0.002 ]

=================== Computing SVM_patient_holdout ===================
Running PFI (5-fold CV, 11 slices of 25 pts, N=3000)...
  Fold 1/5 complete.
  Fold 2/5 complete.
  Fold 3/5 complete.
  Fold 4/5 complete.
  Fold 5/5 complete.
PFI finished in 4.3s.
Saved imp_SVM_patient_holdout.npy with scores: [5.300e-03 1.000e-03 3.300e-03 2.300e-03 1.430e-02 5.163e-01 1.050e-01
 1.300e-03 3.000e-04 2.000e-03 1.300e-03]

=================== Computing RF_beat_holdout ===================
Running PFI (5-fold CV, 11 slices of 25 pts, N=3000)...
  Fold 1/5 complete.
  Fold 2/5 complete.
  Fold 3/5 complete.
  Fold 4/5 complete.
  Fol